#### Dashboard에서 롱폼 성과 예측 및 댓글 내 긍정반응 비율 예측 모델링 페이지 정확도 확인용

In [1]:
# 1. 라이브러리 호출
import pandas as pd
from pathlib import Path

In [2]:
# 2. 파일 경로 설정
BASE = Path.cwd() / 'Dashboard'   # 노트북 기준 한 단계 하위 폴더 (공통)

# 롱폼 영상_성과 (성공확률 예측)
LONGFORM_TEST_PATH = BASE / 'longform' / 'test_set_with_predictions.csv'

# 롱폼 영상 댓글 (긍정비율 예측)
LONGFORM_COMMENT_TEST_PATH = BASE / 'longform_comment' / 'test_data.csv'

# 댓글 테스트셋 video_id 매핑
COMMENT_VIDEO_ID_MAP_PATH = BASE / 'longform_comment' / 'test_video_id_map.csv'

In [3]:
# 3. 데이터 로드
video_test_df    = pd.read_csv(LONGFORM_TEST_PATH, encoding='utf-8-sig')
comment_test_df  = pd.read_csv(LONGFORM_COMMENT_TEST_PATH, encoding='utf-8-sig')
video_id_map     = pd.read_csv(COMMENT_VIDEO_ID_MAP_PATH, encoding='utf-8-sig')

print(f'영상 테스트셋: {video_test_df.shape}')
print(f'댓글 테스트셋: {comment_test_df.shape}')
print(f'video_id 매핑: {video_id_map.shape}')

영상 테스트셋: (219, 6)
댓글 테스트셋: (122, 785)
video_id 매핑: (122, 1)


In [4]:
# 4. 댓글 테스트셋에 video_id 복원
assert len(video_id_map) == len(comment_test_df), \
    f"길이 불일치: video_id_map={len(video_id_map)}, comment_test_df={len(comment_test_df)}"

comment_test_df = pd.concat(
    [video_id_map.reset_index(drop=True), comment_test_df.reset_index(drop=True)],
    axis=1
)

print(f'video_id 복원 완료: {comment_test_df.shape}')
print(comment_test_df[['video_id']].head())

video_id 복원 완료: (122, 786)
      video_id
0  7RfBRiMRV-k
1  dHqE3E846Pk
2  K8ve8v0OCy4
3  xVi2p0r1cKw
4  B6_wqyNuliQ


### 4번 코드 문법 설명

1. `assert`

```python
assert 조건, "에러 메시지"
```

- 조건이 `True`면 통과
- 조건이 `False`면 에러를 내며 멈춤

여기서는 두 데이터프레임의 행 개수가 같은지 미리 확인하는 안전장치다. 길이가 다르면 다음 단계에서 행 순서가 어긋난 채로 합쳐질 위험이 있어서, 미리 차단한다.

---

2. `\` (백슬래시, 줄바꿈)

코드 한 줄이 너무 길 때 다음 줄로 이어 쓰는 문법.

```python
assert A == B, \
    "메시지"
```

위 코드는 실제로는 **한 문장**이다.


---

3. `pd.concat([...], axis=1)`

| 옵션 | 의미 |
|---|---|
| `axis=0` | 위아래로 행을 쌓음 |
| `axis=1` | **옆으로(가로 방향, 컬럼 방향) 붙임** |

```python
pd.concat([video_id_map, comment_test_df], axis=1)
```

→ `video_id_map`(1개 컬럼)을 `comment_test_df`(785개 컬럼) 앞에 붙여 786개 컬럼의 새 데이터프레임 생성.

In [5]:
# 5. 영상 테스트셋 & 댓글 테스트셋 교집합 찾기

video_ids = set(video_test_df['video_id']) # 데이터프레임의 컬럼 값을 "집합(set)"으로 변환 -> 집합은 중복을 자동으로 제거함
comment_ids = set(comment_test_df['video_id'])
intersection_ids = video_ids & comment_ids # 교집합을 찾는 코드

print(f'영상 테스트셋: {len(video_ids)}개')
print(f'댓글 테스트셋: {len(comment_ids)}개')
print(f'교집합: {len(intersection_ids)}개')

영상 테스트셋: 219개
댓글 테스트셋: 122개
교집합: 29개


In [6]:
# 6. 교집합 영상 필터링 및 저장

intersection_df = video_test_df[video_test_df['video_id'].isin(intersection_ids)].copy()

OUT_PATH = BASE / 'intersection_test_videos.csv'
intersection_df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print(f'[SAVE] {len(intersection_df)}개 저장 → {OUT_PATH.name}')
print(intersection_df[['video_id', 'title', 'grade', 'pred_proba', 'pred_label', 'correct']].to_string())

[SAVE] 29개 저장 → intersection_test_videos.csv
        video_id                                                                                       title  grade  pred_proba  pred_label  correct
0    qQoIXf2eJQk                                  [🤎  빼빼로왔CU🤎  ] 일동 주목 ! 빼빼로데이 기념 한정판 콜라보 출시✨  |신상왔CU 빼빼로데이편      0    0.021250           0     True
1    vRD_tfpK96E                                                               [경리회계] 다우오피스 경리회계 초기 설정 퀵 가이드      1    0.754930           1     True
7    eihPljvqbzw                                                               [YoungForce] 신한DS E-Sports 대회      1    0.935654           1     True
8    N3ATBSaCZ70                           날 막지 마! 술안주 특집🔥 오늘도 달려 달려~🍻 | 신상왔씨유 7월 4주차 #신상품 #백걸리 #짐빔하이볼 #자이언트      0    0.011400           0     True
19   L9yrpjwRbu4                                                                           개인사업자 부가세 확정신고 방법      1    0.332842           0    False
23   EPEMnwt4hjM                  ChatGPT 어디까지 써봤니? 스타트업을 위한 

In [7]:
# 7. 교집합 영상 예측 정확도 계산

accuracy = (intersection_df['correct'] == True).mean()
print(f'전체 정확도: {accuracy:.4f} ({accuracy*100:.1f}%)')

# grade(실제 결과)별로 나눠서 확인
g0 = intersection_df[intersection_df['grade'] == 0]
g1 = intersection_df[intersection_df['grade'] == 1]

print(f'실패 영상(grade=0) {len(g0)}개 중 정확도: {(g0["correct"]==True).mean():.4f}')
print(f'성공 영상(grade=1) {len(g1)}개 중 정확도: {(g1["correct"]==True).mean():.4f}')

전체 정확도: 0.7241 (72.4%)
실패 영상(grade=0) 6개 중 정확도: 0.8333
성공 영상(grade=1) 23개 중 정확도: 0.6957


In [8]:
# 8. MLP 예측 결과 불러오기 + RMSE 검증

mlp_pred_df = pd.read_csv(BASE / 'longform_comment' / 'mlp_test_predictions.csv', encoding='utf-8-sig')

print(f'MLP 예측 결과: {mlp_pred_df.shape}')

# 전체 122개 기준 RMSE (보고된 0.2580과 비교)
overall_rmse = ((mlp_pred_df['actual_positive_ratio'] - mlp_pred_df['predicted_positive_ratio']) ** 2).mean() ** 0.5
print(f'전체 122개 RMSE: {overall_rmse:.4f}')

MLP 예측 결과: (122, 3)
전체 122개 RMSE: 0.2580


In [9]:
# 9. 교집합 29개 영상만 떼어서 MLP 오차 확인

# RMSE (Root Mean Squared Error): 오차를 제곱한 뒤 평균을 내고, 다시 제곱근을 씌운 값
#   -> 오차를 제곱하기 때문에 큰 오차에 더 민감하게 반응함 (이상치에 취약)
# MAE (Mean Absolute Error): 오차의 절댓값을 그대로 평균낸 값
#   -> 모든 오차를 동등한 비중으로 다룸
# RMSE가 MAE보다 크게 나오면, 일부 영상에서 유독 크게 틀린 케이스가 섞여 있다는 신호임

mlp_intersection_df = mlp_pred_df[mlp_pred_df['video_id'].isin(intersection_ids)].copy()

mlp_intersection_df['abs_error'] = (
    mlp_intersection_df['actual_positive_ratio'] - mlp_intersection_df['predicted_positive_ratio']
).abs()

subset_rmse = ((mlp_intersection_df['actual_positive_ratio'] - mlp_intersection_df['predicted_positive_ratio']) ** 2).mean() ** 0.5
subset_mae  = mlp_intersection_df['abs_error'].mean()

print(f'교집합 {len(mlp_intersection_df)}개 RMSE: {subset_rmse:.4f}')
print(f'교집합 {len(mlp_intersection_df)}개 MAE:  {subset_mae:.4f}')
print()
print(mlp_intersection_df[['video_id', 'actual_positive_ratio', 'predicted_positive_ratio', 'abs_error']].to_string())

교집합 29개 RMSE: 0.3278
교집합 29개 MAE:  0.2667

        video_id  actual_positive_ratio  predicted_positive_ratio  abs_error
0    7RfBRiMRV-k               0.666667                  0.427756   0.238911
1    dHqE3E846Pk               0.928571                  0.372953   0.555618
10   N3ATBSaCZ70               0.600000                  0.783433   0.183433
14   L9yrpjwRbu4               1.000000                  0.257601   0.742399
15   h4DQ3jX837k               0.890756                  0.997134   0.106377
19   EPEMnwt4hjM               1.000000                  0.609117   0.390883
22   dEQVKe1cmq4               1.000000                  0.783525   0.216475
24   fkVYaCJDwZ8               1.000000                  0.760211   0.239789
25   1pjtNhI_yBA               0.997080                  0.922244   0.074837
26   gDb-QzYCZmA               0.750000                  0.867231   0.117231
29   SmGlcTfPDqQ               0.400000                  0.824093   0.424093
33   b22TlxK3bwk               0.

In [10]:
# 10. 오차가 큰 영상 순으로 정렬해서 확인

sorted_df = mlp_intersection_df.sort_values('abs_error', ascending=False)
print(sorted_df[['video_id', 'actual_positive_ratio', 'predicted_positive_ratio', 'abs_error']].to_string())

        video_id  actual_positive_ratio  predicted_positive_ratio  abs_error
14   L9yrpjwRbu4               1.000000                  0.257601   0.742399
121  TVrmCVVRlso               0.000000                  0.668704   0.668704
1    dHqE3E846Pk               0.928571                  0.372953   0.555618
62   vRD_tfpK96E               1.000000                  0.505194   0.494806
36   _y4Ah3-Nj9I               1.000000                  0.538933   0.461067
99   veeA__h-X_8               0.333333                  0.777446   0.444113
29   SmGlcTfPDqQ               0.400000                  0.824093   0.424093
111  HkzWAebyook               0.428571                  0.841965   0.413393
71   rL5NXOICswI               0.333333                  0.735888   0.402555
19   EPEMnwt4hjM               1.000000                  0.609117   0.390883
45   1eCF3d3XmPI               1.000000                  0.731966   0.268034
24   fkVYaCJDwZ8               1.000000                  0.760211   0.239789

---
---
## YouTrack 롱폼 성과 예측 모델 — 실제 검증 과정

#### 🔺배경

테스트셋 기준 XGBoost 모델의 PR-AUC는 0.9522, MLP 모델의 RMSE는 0.2580이었다. 이 수치들이 실제로 어느 정도 신뢰할 수 있는지 확인하기 위해, 두 모델 모두 학습 때 보지 않은 테스트셋 영상 중 일부를 골라 실제 결과와 예측을 직접 비교하는 검증을 진행했다.

#### 🔺왜 학습 데이터로 검증하면 안 되는가

처음에는 학습에 사용했던 영상으로 검증하려 했으나, 이는 데이터 누수(Data Leakage)에 해당한다. 모델이 이미 학습 과정에서 본 데이터이므로 정확히 맞히는 것이 당연하고, 모델의 실제 일반화 성능을 보여주지 못한다. 따라서 `train_test_split`으로 분리해둔 테스트셋(학습에 쓰이지 않은 영상)을 검증 대상으로 삼았다.

#### 🔺검증 대상 선정 과정

**1. 영상 성과 예측 모델(XGBoost) 테스트셋 확인**

`longform_analysis_with_embedding.ipynb`의 `train_test_split(test_size=0.2, stratify=y, random_state=42)`로 분리된 테스트셋 219개를 확인하고, `video_id`·`title`·`grade`(실제 결과)·`pred_proba`(예측 확률)·`pred_label`(예측 결과)·`correct`(일치 여부) 컬럼으로 정리해 `test_set_with_predictions.csv`로 저장했다.

**2. 댓글 긍정 반응 예측 모델(MLP) 테스트셋 확인**

`longform_comment_analysis.ipynb`의 별도 테스트셋 122개를 확인했다. 단, 이 테스트셋에는 `video_id`가 빠져 있어 어떤 영상인지 식별이 불가능한 상태였다.

**3. 댓글 테스트셋에 video_id 복원**

`df_model`(피처만 추출한 데이터프레임)에는 `video_id`가 없었으나, 그 이전 단계인 `df_full`에는 `video_id`가 남아 있었다. `train_test_split` 시 함께 추출한 인덱스(`idx_test`)를 활용해 `df_full.iloc[idx_test]`로 테스트셋 122개에 해당하는 `video_id`를 복원했다.

```python
video_id_map = df_full.iloc[idx_test][['video_id']].reset_index(drop=True)
video_id_map.to_csv('test_video_id_map.csv', index=False, encoding='utf-8-sig')
```

**4. 두 테스트셋의 교집합 추출**

영상 성과 예측 테스트셋(219개)과 댓글 긍정 반응 예측 테스트셋(122개, video_id 복원 완료)의 `video_id`를 기준으로 교집합을 구했다.

```python
video_ids        = set(video_test_df['video_id'])
comment_ids      = set(comment_test_df['video_id'])
intersection_ids = video_ids & comment_ids
```

영상 테스트셋 219개, 댓글 테스트셋 122개에서 교집합은 29개였다. 두 테스트셋 모두 `video_id` 중복이 없음을 사전에 확인해, 매핑 과정에서 오류가 없음을 검증했다.

#### 🔺검증 결과 — 영상 성과 예측 모델 (XGBoost)

교집합 29개 영상에 대해 XGBoost 모델의 실제 예측 결과(`pred_label`)와 실제 결과(`grade`)를 비교했다.

| 구분 | 개수 | 정확도 |
|---|---|---|
| 전체 | 29개 | 72.4% (21/29) |
| 실패 영상(grade=0) | 6개 | 83.3% (5/6) |
| 성공 영상(grade=1) | 23개 | 69.6% (16/23) |

테스트셋 전체(219개) 기준 PR-AUC는 0.9522였으나, 교집합 29개로 좁혀 단순 정확도를 계산하면 72.4%로 나타났다. 성공 영상을 맞히는 정확도(69.6%)가 실패 영상(83.3%)보다 낮게 나타나, 모델이 성공으로 예측했지만 실제로는 실패한 영상이 다수 존재함을 확인했다.

#### 🔺검증 결과 — 댓글 긍정 반응 예측 모델 (MLP)

같은 방식으로 MLP 모델의 예측값(`predicted_positive_ratio`)을 실제값(`actual_positive_ratio`)과 비교했다. 전체 테스트셋(122개) 기준 RMSE는 0.2580으로, 보고된 수치와 정확히 일치함을 확인했다.

| 구분 | 개수 | RMSE | MAE |
|---|---|---|---|
| 전체 | 122개 | 0.2580 | - |
| 교집합 | 29개 | 0.3278 | 0.2667 |

오차가 가장 큰 영상들(L9yrpjwRbu4, TVrmCVVRlso, dHqE3E846Pk 등)을 살펴보면, 실제 긍정 비율이 0 또는 1에 가까운 극단값인 경우가 많았다. 반면 모델의 예측값은 0.3~0.7 사이 중간값으로 수렴하는 경향을 보였다. 이는 학습 데이터에서 극단값 비중이 상대적으로 적어, 모델이 평균값 쪽으로 예측을 수렴시키는 한계를 가질 수 있음을 시사한다.

#### 🔺종합 해석

PR-AUC·RMSE는 전체 테스트셋(219개·122개)에서의 성능을 평가한 지표이고, 정확도·교집합 RMSE는 그중 29개만 따로 떼어 평가한 결과다. 평가 대상의 크기와 방식이 다르기 때문에 두 수치가 다르게 나오는 것은 자연스럽다. 다만 29개처럼 표본이 작을 경우, 한두 개의 예측이 틀리는 것만으로도 결과가 크게 흔들릴 수 있어(XGBoost 기준 1개당 약 3.4%p) 전체 테스트셋에서 나온 결과보다 신뢰도가 낮을 수 있다는 점을 함께 확인했다. 또한 MLP 모델은 극단적인 댓글 반응을 예측하는 데 한계가 있다는 구체적인 약점도 발견했다.